# RWKV-Flood: Lightweight Linear-Attention Flood Segmentation (SCRWKV)

This notebook benchmarks the linear-attention RWKV architecture for real-time flood water segmentation.
- **Linear Complexity**: $O(N)$ computational complexity with Pure PyTorch native execution.
- **Evaluation Standard**: Multi-seed (5 seeds) training on FloodKaggle and AIFloodSense, followed by cross-domain generalization.
- **Lightweight Claim**: Architectural profiling for parameter footprint, model weight size, latency, and FPS.


In [ ]:
# 1. Clone RWKV-Flood repository and install lightweight dependencies
!git clone https://github.com/doantrongthai/rwkv-flood.git
%cd rwkv-flood
!pip install -r requirements.txt


In [ ]:
import os

# Select architecture: 'rsrwkv', 'scrwkv', 'vrwkv', 'u_rwkv', 'seg_rwkv'
MODEL_NAME = "scrwkv"

AUTO_PRETRAINED_MODELS = {'deeplabv3'}
IS_PRETRAINED = MODEL_NAME.lower() in AUTO_PRETRAINED_MODELS
PRETRAINED_FLAG = "--pretrained" if IS_PRETRAINED else ""

IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 100
LR = 0.001
LOSS_NAME = "lovasz_focal"
NUM_CLASSES = 1

MULTISEED = True
SEEDS = [42, 123, 456, 789, 2024]
REP_SEED = 42

def get_checkpoint_path(model, loss, dataset, seed, is_multiseed):
    clean_loss = loss.lower().replace('-', '_')
    candidates = [
        f"outputs/seed_{seed}/{model}_{clean_loss}_{dataset}_s{seed}.pth",
        f"outputs/{model}_{clean_loss}_{dataset}_s{seed}.pth",
        f"outputs/seed_{seed}/{model}_{loss}_{dataset}_s{seed}.pth",
        f"outputs/{model}_{loss}_{dataset}_s{seed}.pth"
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    return candidates[0] if is_multiseed else candidates[1]


In [ ]:
# 2. Profile Architecture & Compute Lightweight Claim Metrics
import torch
from models import get_model
from utils.metrics import measure_inference_time

print("=" * 70)
print(f"LIGHTWEIGHT ARCHITECTURE CLAIM & PROFILING: {MODEL_NAME.upper()}")
print("=" * 70)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
model = get_model(MODEL_NAME, num_classes=NUM_CLASSES, input_size=IMAGE_SIZE).to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
param_size_mb = sum(p.numel() * p.element_size() for p in model.parameters()) / (1024 * 1024)

fps, latency_ms = measure_inference_time(
    model, (1, 3, IMAGE_SIZE, IMAGE_SIZE), device=device, warmup=15, iterations=100
)

print(f"Model Architecture    : {MODEL_NAME.upper()}")
print(f"Total Parameters      : {total_params:,} ({total_params / 1e6:.2f}M)")
print(f"Trainable Parameters  : {trainable_params:,}")
print(f"Model Size (FP32)     : {param_size_mb:.2f} MB")
print(f"Input Resolution      : {IMAGE_SIZE}x{IMAGE_SIZE}")
print(f"Compute Device        : {device.upper()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'})")
print(f"Latency (per frame)   : {latency_ms:.2f} ms")
print(f"Throughput            : {fps:.1f} FPS")
print("=" * 70)
print("LaTeX Paper Summary Table Line:")
print(f"{MODEL_NAME.upper()} & {total_params / 1e6:.2f}M & {param_size_mb:.2f}MB & {latency_ms:.2f} ms & {fps:.1f} FPS \\\\")
print("=" * 70)


In [ ]:
# 3. Multi-Seed Training on FloodKaggle
if MULTISEED:
    seeds_str = " ".join(map(str, SEEDS))
    !python main.py --dataset floodkaggle --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --multiseed --seeds {seeds_str} {PRETRAINED_FLAG} --download
else:
    !python main.py --dataset floodkaggle --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --seed {REP_SEED} {PRETRAINED_FLAG} --download


In [ ]:
# 4. Multi-Seed Training on AIFloodSense
if MULTISEED:
    seeds_str = " ".join(map(str, SEEDS))
    !python main.py --dataset aifloodsense --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --multiseed --seeds {seeds_str} {PRETRAINED_FLAG} --download
else:
    !python main.py --dataset aifloodsense --model {MODEL_NAME} --num_classes {NUM_CLASSES} --loss {LOSS_NAME} --size {IMAGE_SIZE} --epochs {EPOCHS} --batch_size {BATCH_SIZE} --lr {LR} --seed {REP_SEED} {PRETRAINED_FLAG} --download


In [ ]:
# 5. Cross-Dataset Generalization: Source = FloodKaggle -> Targets = [FloodKaggle, AIFloodSense, FloodVN]
weights_fk = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "floodkaggle", REP_SEED, MULTISEED)
!python cross_dataset_eval.py --model {MODEL_NAME} --weights {weights_fk} --source_dataset floodkaggle --target_datasets floodkaggle aifloodsense floodvn --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --download


In [ ]:
# 6. Cross-Dataset Generalization: Source = AIFloodSense -> Targets = [AIFloodSense, FloodKaggle, FloodVN]
weights_afs = get_checkpoint_path(MODEL_NAME, LOSS_NAME, "aifloodsense", REP_SEED, MULTISEED)
!python cross_dataset_eval.py --model {MODEL_NAME} --weights {weights_afs} --source_dataset aifloodsense --target_datasets aifloodsense floodkaggle floodvn --size {IMAGE_SIZE} --num_classes {NUM_CLASSES} --download


In [ ]:
# 7. Bundle all artifacts & Download .zip package
import os
import shutil
from google.colab import files

bundle_dir = f"{MODEL_NAME}_All_Artifacts"
os.makedirs(bundle_dir, exist_ok=True)
output_dir = "outputs"

if os.path.exists(output_dir):
    for item in sorted(os.listdir(output_dir)):
        src = os.path.join(output_dir, item)
        dst = os.path.join(bundle_dir, item)
        if os.path.isdir(src):
            if os.path.exists(dst):
                shutil.rmtree(dst)
            shutil.copytree(src, dst)
        elif os.path.isfile(src):
            shutil.copy2(src, dst)

zip_name = f"{MODEL_NAME}_All_Results.zip"
shutil.make_archive(f"{MODEL_NAME}_All_Results", "zip", bundle_dir)
print(f"[OK] Archive generated: {zip_name} ({os.path.getsize(zip_name) / (1024*1024):.2f} MB)")
try:
    files.download(zip_name)
    print(f"[OK] Download triggered for: {zip_name}")
except Exception as e:
    print(f"[INFO] In non-interactive mode: {e}. File is saved at {os.path.abspath(zip_name)}")
